In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    """Locate the repository root from either the repo or notebooks folder."""
    for folder in [start, *start.parents]:
        if (folder / "data").exists() and (folder / "notebooks").exists():
            return folder
    raise FileNotFoundError("Could not locate the repository root.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DEFAULT_CONFIG

RAW_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "price_history_checks_mar2026.csv"
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "figures"
SYDNEY_POSTCODE_MIN = DEFAULT_CONFIG.sydney_postcode_min
SYDNEY_POSTCODE_MAX = DEFAULT_CONFIG.sydney_postcode_max

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_PATH.exists():
    raise FileNotFoundError(f"Required raw data not found: {RAW_PATH}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data: {RAW_PATH}")


In [ ]:
import pandas as pd

df = pd.read_csv(RAW_PATH)

print(df.shape)
df.head()


In [ ]:
df.columns

In [ ]:
df_clean = df.copy()

df_clean["PriceUpdatedDate"] = pd.to_datetime(df_clean["PriceUpdatedDate"], errors="coerce")
df_clean["Price"] = pd.to_numeric(df_clean["Price"], errors="coerce")

df_clean = df_clean.dropna(subset=["PriceUpdatedDate", "Price"])

df_clean["date"] = df_clean["PriceUpdatedDate"].dt.date

df_clean.head()

In [ ]:
df_clean["FuelCode"].value_counts().head(20)

In [ ]:
fuel_code = "U91"

In [ ]:
sydney = df_clean[
    (df_clean["Postcode"] >= SYDNEY_POSTCODE_MIN) &
    (df_clean["Postcode"] <= SYDNEY_POSTCODE_MAX)
].copy()

print(sydney.shape)
sydney[["Suburb", "Postcode"]].drop_duplicates().head(20)

In [ ]:
sydney_u91 = sydney[sydney["FuelCode"] == "U91"].copy()

print(sydney_u91.shape)
sydney_u91.head()

In [ ]:
daily_avg = (
    sydney_u91
    .groupby("date", as_index=False)
    .agg(
        average_price_cpl=("Price", "mean"),
        station_count=("ServiceStationName", "nunique")
    )
)

daily_avg.head()

In [ ]:
processed_path = PROCESSED_DIR / "sydney_u91_daily_average_mar2026.csv"
daily_avg.to_csv(processed_path, index=False)


In [ ]:
import matplotlib.pyplot as plt

# 1. Load raw data
df = pd.read_csv(RAW_PATH)

# 2. Clean columns
df_clean = df.copy()

df_clean["PriceUpdatedDate"] = pd.to_datetime(df_clean["PriceUpdatedDate"], errors="coerce")
df_clean["Price"] = pd.to_numeric(df_clean["Price"], errors="coerce")
df_clean["Postcode"] = pd.to_numeric(df_clean["Postcode"], errors="coerce")

df_clean = df_clean.dropna(subset=["PriceUpdatedDate", "Price", "Postcode"])
df_clean["date"] = df_clean["PriceUpdatedDate"].dt.date

# 3. Filter Sydney metro and U91
sydney_u91 = df_clean[
    (df_clean["Postcode"] >= SYDNEY_POSTCODE_MIN) &
    (df_clean["Postcode"] <= SYDNEY_POSTCODE_MAX) &
    (df_clean["FuelCode"] == "U91")
].copy()

# 4. Create daily average dataset
daily_avg = (
    sydney_u91
    .groupby("date", as_index=False)
    .agg(
        average_price_cpl=("Price", "mean"),
        station_count=("ServiceStationName", "nunique"),
        observation_count=("Price", "count")
    )
)

# 5. Save processed CSV
processed_path = PROCESSED_DIR / "sydney_u91_daily_average_mar2026.csv"
daily_avg.to_csv(processed_path, index=False)

# 6. Create rolling average
daily_avg["date"] = pd.to_datetime(daily_avg["date"])
daily_avg["rolling_7d_avg"] = daily_avg["average_price_cpl"].rolling(window=7, min_periods=1).mean()

# 7. Plot
plt.figure(figsize=(12, 6))

plt.plot(
    daily_avg["date"],
    daily_avg["average_price_cpl"],
    marker="o",
    linewidth=1.5,
    markersize=4,
    label="Daily average"
)

plt.plot(
    daily_avg["date"],
    daily_avg["rolling_7d_avg"],
    linewidth=2.2,
    label="7-day rolling average"
)

plt.title("Sydney Daily Average U91 Petrol Price Based on FuelCheck Data, March 2026")
plt.xlabel("Date")
plt.ylabel("Average price (cpl)")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()

output_path = FIGURES_DIR / "sydney_u91_daily_average_mar2026.png"
plt.savefig(output_path, dpi=300)
plt.show()

daily_avg.head()
